# GOAL: This file aims to query observed, difference, and template images for the purpose of seeing the before, during, and after of measurements that we suspect are erroneous

In [1]:
from lsst.daf.butler import Butler, Timespan
from lsst.rsp.service import get_siav2_service
from lsst.rsp.utils import get_pyvo_auth
import lsst.afw.display as afwDisplay
from astropy.time import Time

import lsst.geom as geom
from lsst.afw.image import ExposureF
import astropy.units as u
from pyvo.dal.adhoc import DatalinkResults, SodaQuery
from lsst.afw.fits import MemFileManager
import matplotlib.pyplot as plt

#Temp imports to test with
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize
from astropy.io import fits

import numpy as np

In [2]:
#Set the display to use matplotlib
afwDisplay.setDefaultBackend('matplotlib')

#Define the butler and SIA (used to query)
butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

service = get_siav2_service("dp1")
assert service is not None

### Searching for the specific name we should use to query for difference images so that we can set calib_level and dpsubtype properly later

In [3]:
#Searching for the name to use to query for difference images
for dt in sorted(butler.registry.queryDatasetTypes(), key=lambda x: x.name):
    name = dt.name.lower()
    if "diff" in name:
        print(dt.name)

difference_image
forcedPhotDiaObjectDifference_config
forcedPhotDiaObjectDifference_gatherResourceUsage_config
forcedPhotDiaObjectDifference_gatherResourceUsage_log
forcedPhotDiaObjectDifference_gatherResourceUsage_metadata
forcedPhotDiaObjectDifference_log
forcedPhotDiaObjectDifference_metadata
forcedPhotObjectDifference_config
forcedPhotObjectDifference_gatherResourceUsage_config
forcedPhotObjectDifference_gatherResourceUsage_log
forcedPhotObjectDifference_gatherResourceUsage_metadata
forcedPhotObjectDifference_log
forcedPhotObjectDifference_metadata


### This is the old version of the query function; ignore

In [4]:
def queryAndPlot(butler, afwDisplay, imageType, ra, dec, time, band):
    #Define the timespan
    startTime = time - 0.000001
    endTime = time + 0.000001
    time1 = Time(startTime, format="mjd", scale="tai")
    time2 = Time(endTime, format="mjd", scale="tai")
    timespan = Timespan(time1, time2)

    #Perform the query
    dataset_refs = butler.query_datasets(imageType,
                                     where="band.name = :band AND \
                                     visit.timespan OVERLAPS :timespan AND \
                                     visit_detector_region.region OVERLAPS POINT(:ra, :dec)",
                                     bind={"band": band, "timespan": timespan,
                                           "ra": ra, "dec": dec},
                                     order_by=["visit.timespan.begin"])

    visit_image = butler.get(dataset_refs[0])

    #Setup the display
    display = afwDisplay.Display()
    display.scale('asinh', 'zscale')
    display.image(visit_image.image)

    #Get the coordinates of the star in pixels and plot horizontal/vertical lines
    wcs = visit_image.getWcs()
    pixels = wcs.skyToPixel(geom.SpherePoint(ra * geom.degrees, dec * geom.degrees))
    xPixels, yPixels = pixels.getX(), pixels.getY()
    plt.axvline(x=xPixels, color='lime', linestyle='--', linewidth=1)
    plt.axhline(y=yPixels, color='lime', linestyle='--', linewidth=1)

    #Set title
    if(imageType == "difference_image"):
        titleImageType = "diffImg"
    elif(imageType == "visit_image"):
        titleImageType = "visImg"
    else:
        titleImageType = "other"
        
    plt.title(f'''{titleImageType} at ({ra},{dec}) over {time} in band {band}''')

    #Display
    plt.show()

### This is the new/current version of the query function

#### Inputs:
##### -RA/DEC for location
##### -Time in MJD
##### -scalingType: The way that we want to scale our flux measurements in the displayed graph (either linear or asinh)
##### -zLower/zUpper: The upper and lower bounds for scaling purposes; if left as None will automatically use "best fitting" bounds
##### -service/afwDisplay: references being used by the function, I'm honestly not sure if we need to include these, I put them in when I first made the function and never bothered looking into if they were necessary because it's been working
####
#### Functionality:
##### -Searches for an image of the image type provided where the ra/dec provided fall within the image and the time provided falls within the time frame of the image
##### -If the search is not empty, use the first working values from the search (results often contains multiple different iterations of outputs and often times many of these iterations have incomplete/problematic data that will through errors when we try to graph it)
##### -Plot these first working values based off their pixel locations on the image (is proportionate to ra/dec but doesn't display using ra/dec)
##### -Plots are scaled using scalingType, zLower, and zUpper (read above)

In [14]:
def queryCutout(service, afwDisplay, imageType, scalingType, ra, dec, time, zLower, zUpper):
    #Declare the coordinates that we want to search for
    circle = (ra, dec, 0.05)

    #Declare the time we want to search for
    startTime = time - 0.0001
    endTime = time + 0.0001
    time1 = Time(startTime, format="mjd", scale="tai")
    time2 = Time(endTime, format="mjd", scale="tai")

    #Declare the calibration level and the type of image we want to search for
    if(imageType == 'template'):
        calib_level = 3
        dpsubtype = 'lsst.template_coadd'
    elif(imageType == 'difference'):
        calib_level = 3
        dpsubtype = 'lsst.difference_image'
    elif(imageType == 'observed'):
        calib_level = 2
        dpsubtype = 'lsst.visit_image'
    else:
        print(f'''Unsupported image type: {imageType}''')
        return
    
    #Perform the query
    results = service.search(pos=circle, calib_level=calib_level, dpsubtype=dpsubtype, band=622.1e-09, time=(time1, time2))

    #Make sure there actually are results
    # print(len(results))
    if len(results) == 0:
        print(f"No {imageType} data found at RA={ra}, Dec={dec}, MJD={time:.6f}")
        return None, None

    # print(f"{imageType} SIA results for MJD={time}: {len(results)} rows")
    # for index,row in enumerate(results):
    #     print(results[index])
    for idx, row in enumerate(results):
        try:
            #Convert to url
            datalink_url = results[idx].access_url
            dl_result = DatalinkResults.from_result_url(datalink_url, session=get_pyvo_auth())
        
            #Initialize sq as a SodaQuery object for the purpose of doing cutouts
            sq = SodaQuery.from_resource(dl_result,
                                     dl_result.get_adhocservice_by_id("cutout-sync"),
                                     session=get_pyvo_auth())
        
            #Details of the cutout
            spherePoint = geom.SpherePoint(ra*geom.degrees, dec*geom.degrees)
            Radius = 0.0025 * u.deg #This defines how large our image will be
            sq.circle = (spherePoint.getRa().asDegrees() * u.deg,
                     spherePoint.getDec().asDegrees() * u.deg,
                     Radius)
        
            cutout_bytes = sq.execute_stream().read()
            sq.raise_if_error()
            mem = MemFileManager(len(cutout_bytes))
            mem.setData(cutout_bytes, len(cutout_bytes))
            exposure = ExposureF(mem)
        
            #Setup the display
            display = afwDisplay.Display()
            
            if(zLower is not None and zUpper is not None): #If zLower and zUpper passed in, use them. Else automatically calculate it
                display.scale(scalingType, zLower, zUpper, 'Absolute')
            else:
                display.scale(scalingType, 'zscale')
                
            display.image(exposure.image)
        
            #Get the coordinates of the star in pixels and plot horizontal/vertical lines
            wcs = exposure.getWcs()
            pixels  = wcs.skyToPixel(spherePoint)
            xPixels, yPixels = pixels.getX(), pixels.getY()
            plt.axvline(x=xPixels, color='lime', linestyle='--', linewidth=1)
            plt.axhline(y=yPixels, color='lime', linestyle='--', linewidth=1)

            # START OF TEMP CHANGES #
            # bbox = exposure.getBBox()  # Box2I in pixel coords
            # # Pixel corners of the cutout
            # x_min, y_min = bbox.getMinX(), bbox.getMinY()
            # x_max, y_max = bbox.getMaxX(), bbox.getMaxY()
            
            # corners_pix = [
            #     geom.Point2D(x_min, y_min),  # lower left
            #     geom.Point2D(x_max, y_min),  # lower right
            #     geom.Point2D(x_max, y_max),  # upper right
            #     geom.Point2D(x_min, y_max),  # upper left
            # ]
            
            # corners_sky = [wcs.pixelToSky(p) for p in corners_pix]
            
            # # Extract RA/Dec (in degrees)
            # ras  = [sp.getRa().asDegrees()  for sp in corners_sky]
            # decs = [sp.getDec().asDegrees() for sp in corners_sky]
            
            # ra_min, ra_max   = min(ras), max(ras)
            # dec_min, dec_max = min(decs), max(decs)
            
            # print(f"RA range:  {ra_min:.8f} – {ra_max:.8f} deg")
            # print(f"Dec range: {dec_min:.8f} – {dec_max:.8f} deg")
            # END OF TEMP CHANGES #
        
            #Plot the title
            plt.title(f'''{imageType} image at ({ra},{dec}) over {time} in band r''')
        
            plt.show()
        
            # img_array = exposure.image.array
            zLower, zUpper = afwDisplay.getZScale(exposure.image)
        
            # print(wcs.getCdMatrix())
        
            return zLower, zUpper
        except Exception as e:
            print(f'''Query failed for result[{idx}]''')

    print('Queries failed for all results')
    return None, None

## This is just a helper function that calls queryCutouts 3 times in a row for both observed and difference images
#### We've been looking specifically into erroneous measurements and their surrounding measurements so this fits our particular application
#### "asinhConsistent" means using asinh scaling and using the first observation time's zLower and zUpper to scale the next two observation times
#### "linearConsistent" means using linear scaling and using the first observation time's zLower and zUppwer to scale the next two observation times
#### "asinh" means using asinh scaling and letting autoscaling figure out the zLower and zUpper for all observation times
#### NOTE: asinhConsistent has been scaling incorrectly and doesn't properly handle upper/lower bounds; until it's fixed, it's kind of useless

In [6]:
def querySequence(sequenceType, service, afwDisplay, ra, dec, previousTime, errTime, nextTime):
    if(sequenceType == 'asinhConsistent'):
        #Previous Value
        print('Previous Value')
        zLowerDiff, zUpperDiff = queryCutout(service, afwDisplay, 'difference', 'asinh', ra, dec, previousTime, None, None)
        zLowerObs, zUpperObs = queryCutout(service, afwDisplay, 'observed', 'asinh', ra, dec, previousTime, None, None)

        #Erroneous Value
        print('Erroneous Value')
        queryCutout(service, afwDisplay, 'difference', 'asinh', ra, dec, errTime, zLowerDiff, zUpperDiff)
        queryCutout(service, afwDisplay, 'observed', 'asinh', ra, dec, errTime, zLowerObs, zUpperObs)

        #Next Value
        print('Next Value')
        queryCutout(service, afwDisplay, 'difference', 'asinh', ra, dec, nextTime, zLowerDiff, zUpperDiff)
        queryCutout(service, afwDisplay, 'observed', 'asinh', ra, dec, nextTime, zLowerObs, zUpperObs)

    elif(sequenceType == 'linearConsistent'):
        #Previous Value
        print('Previous Value')
        zLowerDiff, zUpperDiff = queryCutout(service, afwDisplay, 'difference', 'linear', ra, dec, previousTime, None, None)
        zLowerObs, zUpperObs = queryCutout(service, afwDisplay, 'observed', 'linear', ra, dec, previousTime, None, None)

        #Erroneous Value
        print('Erroneous Value')
        queryCutout(service, afwDisplay, 'difference', 'linear', ra, dec, errTime, zLowerDiff, zUpperDiff)
        queryCutout(service, afwDisplay, 'observed', 'linear', ra, dec, errTime, zLowerObs, zUpperObs)

        #Next Value
        print('Next Value')
        queryCutout(service, afwDisplay, 'difference', 'linear', ra, dec, nextTime, zLowerDiff, zUpperDiff)
        queryCutout(service, afwDisplay, 'observed', 'linear', ra, dec, nextTime, zLowerObs, zUpperObs)
        
    elif(sequenceType == 'asinh'):
        #Previous Value
        print('Previous Value')
        queryCutout(service, afwDisplay, 'difference', 'asinh', ra, dec, previousTime, None, None)
        queryCutout(service, afwDisplay, 'observed', 'asinh', ra, dec, previousTime, None, None)

        #Erroneous Value
        print('Erroneous Value')
        queryCutout(service, afwDisplay, 'difference', 'asinh', ra, dec, errTime, None, None)
        queryCutout(service, afwDisplay, 'observed', 'asinh', ra, dec, errTime, None, None)

        #Next Value
        print('Next Value')
        queryCutout(service, afwDisplay, 'difference', 'asinh', ra, dec, nextTime, None, None)
        queryCutout(service, afwDisplay, 'observed', 'asinh', ra, dec, nextTime, None, None)

    else:
        print(f'''Sequence type: {sequenceType} is not an accepted argument''')

    print('Template Image')
    queryCutout(service, afwDisplay, 'template', 'asinh', ra, dec, errTime, None, None)

### TESTING - Currently not working properly? I'm not sure what happened. I didn't change the code but now suddenly all queries are failing. Maybe the dataset failed or maybe there's some technical difficulties on my end?
#### Test inputs - Refer to "Difference Image Querying with the Butler.pdf" in the "Documents" folder in the GitHub repo

In [15]:
#Object 1
ra = 39.807885
dec = -34.543669
time1 = 60650.178708
time2 = 60650.179128
time3 = 60650.179542

#Object 2
# ra = 40.097735
# dec = -34.570817
# time1 = 60639.249554
# time2 = 60639.250143
# time3 = 60639.250758

#Object 3
# ra = 40.024868
# dec = -34.53356
# time1 = 60639.244622
# time2 = 60639.245205
# time3 = 60639.245811

#Object 4
# ra = 40.233196
# dec = -34.585157
# time1 = 60639.248370
# time2 = 60639.248962
# time3 = 60639.249554

#Object 5
# ra = 40.119399
# dec = -34.507458
# time1 = 60639.249554
# time2 = 60639.250143
# time3 = 60639.250758

#Object 6
# ra = 40.033213
# dec = -34.570647
# time1 = 60639.244622
# time2 = 60639.245205
# time3 = 60639.245811
# time4 = 60639.246978

#Object 7-1
# ra = 40.046007
# dec = -34.550136
# time1 = 60639.244622
# time2 = 60639.245811
# time3 = 60639.246978

#Object 7-2
# ra = 40.046007
# dec = -34.550136
# time1 = 60639.251930
# time2 = 60639.252513
# time3 = 60639.253099

#### Testing linearConsistent and asinh sequences for the current test values

In [16]:
querySequence('linearConsistent', service, afwDisplay, ra, dec, time1, time2, time3)

Previous Value
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
Erroneous Value
Query failed for result[0]
Queries failed for all results
Query failed for result[0]
Queries failed for all results
Next Value
Query failed for result[0]
Queries failed for all results
Query failed for result[0]
Queries failed for all results
Template Image
Query failed for result[0]
Query failed for result[1]
Queries failed for all results


In [9]:
querySequence('asinh', service, afwDisplay, ra, dec, time1, time2, time3)

Previous Value
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
Erroneous Value
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
Next Value
Query failed for result[0]
Queries failed for all results
Query failed for result[0]
Queries failed for all results
Template Image
Query failed for result[0]
Query failed for result[1]
Queries failed for all results
